# 05 (Colab) · Enhancement 1: DA3 Self-Audit, original DA3 vs DA3 + Self-Audit
Same as `05_self_audit` (same scripts, same results, same record format), for **Google Colab**.
What it measures: `docs/da3_enhancement_report.md` §4.2; how to save and compare the results:
`docs/ENHANCEMENTS_RUNBOOK.md` (§2b for Colab).

**Before running**
1. **Runtime → Change runtime type → T4 GPU** (this notebook asks for it; check it is selected).
2. Private repo only: left bar → **key icon (Secrets)** → add `GITHUB_TOKEN` and switch on
   **Notebook access**.
3. **Runtime → Run all**, and allow Google Drive access when asked: every finished model is copied
   to Drive (`DRIVE_DIR`), because Colab deletes `/content` at the end of the session.

**Colab limits.** Free sessions disconnect when idle or after a few hours, have ~12 GB of RAM and
2 CPU cores (the TSDF scoring is CPU work and is slower than on Kaggle).
- Start with the quick check (`MAX_SCENES = 2`, the default).
- For full runs, use **one model per session**. If a session is likely to be cut, split it:
  `PHASE = "infer"` with `KEEP_DUMP_ON_DRIVE = True` (the GPU part; its files go to Drive), then a
  new session with `PHASE = "analyse"` and `RESUME_FROM_DRIVE = True`.
- Use the same GPU type (T4) for all runs you compare.

In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Runtime > Run all again"

In [ ]:
# --- repo setup (Google Colab) -----------------------------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"   # fail fast instead of waiting for a password
os.chdir("/content")
url = REPO_URL
try:  # private repo: left bar > key icon (Secrets) > add GITHUB_TOKEN, switch on "Notebook access"
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    print("no GITHUB_TOKEN secret: cloning without a token (works only for a public repo)")
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/content/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
!git log -1 --format="commit %H  %cd"

In [ ]:
# --- official DA3 repo (benchmark harness), pinned commit ---------------------------------------
import os, subprocess
if not os.path.exists("/content/Depth-Anything-3"):
    subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
                    "/content/Depth-Anything-3"], check=True)
subprocess.run(["git", "-C", "/content/Depth-Anything-3", "checkout", "3d835ec1a5802d64a8b8b15f817a1ab54809bfe4"], check=True)
OFFICIAL_SRC = "/content/Depth-Anything-3/src"
!pip install -q -r requirements-eval.txt
# Open3D self-test: does TSDF fusion (depth maps -> mesh) work in this environment?
!python scripts/check_open3d.py --out /content/open3d_check.json

In [ ]:
import json
chk = json.load(open("/content/open3d_check.json"))
assert chk["legacy_tsdf_ok"], ("Open3D TSDF fusion is broken here, so reconstruction metrics would be wrong. "
                               "Stop and send open3d_check.json (or run 05_self_audit on Kaggle).")
print("Open3D TSDF OK:", chk["open3d"])

### Configuration

In [ ]:
MODELS = ["da3-small"]                      # full runs: ONE model per session ("da3-small", then "da3-base")
DATASETS = ["hiroom", "7scenes", "dtu64"]
MAX_SCENES = 2        # 2 = quick end-to-end check (labelled "limited"); 0 = all scenes (the real numbers)
PHASE = "all"         # "all" | "infer" (GPU: baseline run + dump) | "analyse" (CPU work, resumes from Drive)
DRIVE_DIR = "/content/drive/MyDrive/DA3/self_audit_colab"   # results saved here; "" = no Drive (download at the end)
KEEP_DUMP_ON_DRIVE = False   # True: also save the harness exports + dumps (several GB) so PHASE="analyse" can resume
RESUME_FROM_DRIVE = False    # True (with PHASE="analyse"): copy the saved exports + dumps back from Drive first
FUSION_WORKERS = 2           # parallel TSDF fusions (Colab has ~12 GB RAM; Kaggle uses 4)
WORK = "/content/eval_self_audit"           # harness exports + audit dumps (large)
OUT = "/content/self_audit"                 # figures, CSV, summaries
RESULTS = f"{OUT}/results"                  # standard result records (commit these)
BENCH = "/content/da3bench"
DS = " ".join(DATASETS)
def tag_for(m):
    return f"sa_{m}_{'-'.join(DATASETS)}" + (f"_limited{MAX_SCENES}" if MAX_SCENES else "")

import shutil
def to_drive(src, name, ignore=None):
    """Copy a folder to DRIVE_DIR/<name> (merged with what is already there)."""
    if DRIVE_DIR and os.path.exists(src):
        shutil.copytree(src, f"{DRIVE_DIR}/{name}", dirs_exist_ok=True, ignore=ignore)
        print("saved to Drive:", f"{DRIVE_DIR}/{name}")
NO_BENCH_LINK = shutil.ignore_patterns("benchmark_dataset")   # never copy the benchmark data itself

os.makedirs(RESULTS, exist_ok=True)
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
if RESUME_FROM_DRIVE:
    assert DRIVE_DIR and os.path.exists(f"{DRIVE_DIR}/eval_self_audit"), "nothing to resume: run PHASE='infer' with KEEP_DUMP_ON_DRIVE=True first"
    shutil.copytree(f"{DRIVE_DIR}/eval_self_audit", WORK, dirs_exist_ok=True)
    if os.path.exists(f"{DRIVE_DIR}/self_audit/results"):
        shutil.copytree(f"{DRIVE_DIR}/self_audit/results", RESULTS, dirs_exist_ok=True)
    print("resumed from Drive:", sorted(os.listdir(RESULTS)))
print({m: tag_for(m) for m in MODELS})

### Data: DA3-BENCH subsets (HF `depth-anything/DA3-BENCH`)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
os.makedirs(BENCH, exist_ok=True)
for d in DATASETS:
    if not os.path.exists(f"{BENCH}/{d}"):
        z = hf_hub_download("depth-anything/DA3-BENCH", f"{d}.zip", repo_type="dataset", local_dir="/content/zips")
        zipfile.ZipFile(z).extractall(BENCH); os.remove(z)
!du -sh {BENCH}/*

### 1 · Original DA3 on the paper protocol + Self-Audit dump (GPU)
Released harness, camera-head poses (API default). `--dump-audit` also saves, next to each scene's
export, the ray-map poses (official RANSAC routine and our DLT), a pooled ray map and the E_c
re-pass (DA3 conditioned on its own cameras). The exported depth / poses are unchanged.

In [ ]:
if PHASE in ("all", "infer"):
    for m in MODELS:
        t = tag_for(m)
        !python scripts/eval_bench.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK} \
            --weights official:{m} --datasets {DS} --modes pose recon_unposed --max-scenes {MAX_SCENES} \
            --tag {t} --enhancement baseline --variant head --dump-audit --results-root {RESULTS} \
            --fusion-workers {FUSION_WORKERS}
        to_drive(RESULTS, "self_audit/results")
        if KEEP_DUMP_ON_DRIVE:
            to_drive(WORK, "eval_self_audit", ignore=NO_BENCH_LINK)
!du -sh {WORK}

### 2 · Self-Audit analysis (CPU)
Signals → per-view GT errors → audit model (leave-one-dataset-out) → A and C metrics, then the
pose-source variants are written with the **same depth** and scored by the harness (`--eval-only`).

In [ ]:
if PHASE in ("all", "analyse"):
    for m in MODELS:
        t = tag_for(m)
        !python scripts/self_audit_eval.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK} \
            --baseline-tag {t} --weights official:{m} --datasets {DS} --max-scenes {MAX_SCENES} \
            --results-root {RESULTS} --out-dir {OUT}/{t} --fusion-workers {FUSION_WORKERS}
        to_drive(OUT, "self_audit")   # summary, figures, CSV and records of this model

### 3 · Results: original DA3 vs DA3 + Self-Audit

In [ ]:
from IPython.display import Image, Markdown, display
for m in MODELS:
    p = f"{OUT}/{tag_for(m)}/summary.md"
    display(Markdown(open(p).read()) if os.path.exists(p) else Markdown(f"_missing: {p}_"))

In [ ]:
import matplotlib.pyplot as plt
from da3.compare import load_records
from da3.compare.plots import sec6_delta_bars
recs = load_records(RESULTS)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, lab in zip(axes, ["AUC@3", "AUC@30", "F1 w/o p."]):
    sec6_delta_bars(recs, lab, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
for m in MODELS:
    for f in ["risk_coverage.png", "depth_reliability.png", "tc_vs_gt_sigma.png", "audit_coefficients.png", "audit_examples.png"]:
        p = f"{OUT}/{tag_for(m)}/{f}"
        if os.path.exists(p):
            print(m, "·", f); display(Image(p, width=1100))

### 4 · Go/no-go gates (report §4.2)

In [ ]:
for r in load_records(RESULTS):
    if r["enhancement"] == "self_audit" and "gates" in r["extra"]:
        print(f"== {r['weights']} ({r['tag']})")
        for name, g in r["extra"]["gates"].items():
            print(f"  {g['result']:5s} {name}: {g['value']}")

### 5 · Save
`self_audit_results.zip` (summaries, figures, records) goes to Drive (or is downloaded). Then copy
`self_audit/results/*` into the repo's `experiments/results/` and commit, as with the Kaggle run.

In [ ]:
!cd /content && rm -f self_audit_results.zip && zip -qr self_audit_results.zip self_audit && ls -la self_audit_results.zip
!find {RESULTS} -name "*.json" | sort
if DRIVE_DIR:
    shutil.copy("/content/self_audit_results.zip", DRIVE_DIR)
    print("saved to Drive:", f"{DRIVE_DIR}/self_audit_results.zip")
else:
    from google.colab import files
    files.download("/content/self_audit_results.zip")